In [0]:
%pip install -q langchain langchain-databricks langgraph

In [0]:
dbutils.library.restartPython()

In [0]:
# P17: working agent for customer-support
from langchain.tools import tool
from langchain_databricks import ChatDatabricks
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain_core.messages.tool import ToolMessage
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

# Define the state schema
class AgentState(TypedDict):
    order: dict
    messages: Annotated[list, add_messages]

#1: Define our single business tool
@tool
def cancel_order(order_id: str) -> str:
    """Cancel an order that hasn't shipped."""
    #(Here you'd call your real backend API)
    return f"Order {order_id} has been cancelled."

# Create the LLM once and bind the tool
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4-5", temperature=0).bind_tools([cancel_order])

#2: The agent "brain": invoke LLM, run tool, then invoke LLM again
def call_model(state):
    msgs = state["messages"]
    order = state.get("order", {"order_id": "UNKNOWN"})

    #sytem prompt tells the model exactly what to do
    prompt = (
        f'''You are an ecommerce support agent.
        ORDER ID: {order['order_id']}
        If the customer asks to cancel, call cancel_order(order_id)
        and then send a simple confirmation.
        Otherwise, just respond normally.'''
    )
    full = [SystemMessage(prompt)] + msgs
    #1st LLM pass: decides wether to call our tool
    first = llm.invoke(full)
    out = [first]

    if getattr(first, "tool_calls", None):
        #run the cancel_order tool
        tc = first.tool_calls[0]
        result = cancel_order.invoke(tc["args"])
        out.append(ToolMessage(content=result, tool_call_id=tc["id"]))
        #2nd LLM pass: generate the final confirmation text
        second = llm.invoke(full + out)
        out.append(second)
    return {"messages": out}

#3: Write it all up in a StateGraph
def construct_graph():
    graph = StateGraph(AgentState)
    graph.add_node("assistant", call_model)
    graph.set_entry_point("assistant")
    return graph.compile()

graph = construct_graph()

if __name__ == "__main__":
    example_order = {"order_id": "A12345"}
    convo = [HumanMessage(content="Hi, I want to cancel my order A12345")]
    result = graph.invoke({"order": example_order, "messages": convo})
    for msg in result["messages"]:
        print(f"{msg.type}: {msg.content}")